In [ ]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [ ]:
dataset = pd.read_csv('training.1600000.processed.noemoticon.csv', encoding='ISO-8859-1')

In [ ]:
dataset.head()

,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D"
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew


In [ ]:
col_names = ['target', 'ids', 'date', 'flag', 'user', 'text']
dataset.columns = col_names
dataset.head()

,target,ids,date,flag,user,text
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew


In [ ]:
dataset.shape

(1599999, 6)

In [ ]:
dataset.isnull().sum()

,0
target,0
ids,0
date,0
flag,0
user,0
text,0


In [ ]:
dataset['target'].value_counts()

,count
target,
4,800000
0,799999


In [ ]:
dataset['target']=dataset['target'].map({4:1, 0:0})

In [ ]:
dataset['target'].value_counts()

,count
target,
1,800000
0,799999


In [ ]:
# Stemming

stremmer = PorterStemmer()

def stremming(content):
  stremmed_content = re.sub('[^a-zA-Z]',' ',content) #removing not a-z and A=Z
  stremmed_content = stremmed_content.lower()
  stremmed_content = stremmed_content.split()
  stremmed_content = [stremmer.stem(word) for word in stremmed_content if not word in stopwords.words('english')]
  stremmed_content = ' '.join(stremmed_content)
  return stremmed_content

In [ ]:
dataset['text'] = dataset['text'].apply(stremming)

In [ ]:
dataset.head()

,target,ids,date,flag,user,text
0,0.0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,upset updat facebook text might cri result sch...
1,0.0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,kenichan dive mani time ball manag save rest g...
2,0.0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,whole bodi feel itchi like fire
3,0.0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,nationwideclass behav mad see
4,0.0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,kwesidei whole crew


In [ ]:
x = dataset['text']
y = dataset['target']

In [ ]:
# splitting the dataset
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2)

In [ ]:
# convert textual data to numerical data
vectorizer = TfidfVectorizer()
x_train = vectorizer.fit_transform(x_train)
x_test = vectorizer.transform(x_test)

In [ ]:
print(x_train)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 9451285 stored elements and shape (1279999, 460555)>
  Coords	Values
  (0, 101428)	0.776106511775454
  (0, 435323)	0.25795527704002896
  (0, 160320)	0.3121253833533031
  (0, 405465)	0.30300944435328037
  (0, 66697)	0.3766706507364339
  (1, 241501)	0.5940640998201837
  (1, 437481)	0.8044177057380294
  (2, 12095)	0.23036388773072908
  (2, 136138)	0.24374957067347808
  (2, 137887)	0.2309633029863662
  (2, 78963)	0.6392177656235684
  (2, 47931)	0.2285282845469903
  (2, 93796)	0.13649957077984357
  (2, 444838)	0.143958429400076
  (2, 365159)	0.23006205656971884
  (2, 297569)	0.3706694800198963
  (2, 302597)	0.29708219091176463
  (2, 399829)	0.15725437757061775
  (2, 420103)	0.17516479966347792
  (3, 353954)	0.26672998005063203
  (3, 170022)	0.2539587120476497
  (3, 21200)	0.31953329131967634
  (3, 374815)	0.25983295391455086
  (3, 48723)	0.3790033032810089
  (3, 452673)	0.23991670863233922
  :	:
  (1279997, 145761)	0.150676354612

In [ ]:
# Check again for NaN values
print(np.any(np.isnan(y_train)))  # Should print False


True


In [ ]:
# Testing the model
y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Assuming 'stremmer' should be 'stemmer'
stemmer = PorterStemmer()

def predict_sentiment(text):
    text = re.sub('[^a-zA-Z]', '', text)  # Removing non-alphabetic characters
    text = text.lower()  # Convert text to lowercase
    text = text.split()  # Split text into words
    text = [stemmer.stem(word) for word in text if word not in stopwords.words('english')]  # Remove stopwords and apply stemming
    text = ' '.join(text)  # Join words back into a single string
    text = vectorizer.transform([text])  # Vectorize the cleaned text

    prediction = model.predict(text)  # Make prediction using the model

    # Check prediction result
    if prediction == 0:
        return "Negative"
    else:
        return "Positive"


In [ ]:
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

# Sample training data
X_train = ["I am happy", "I am sad", "I love you", "I hate you", "I am excited"]
y_train = [1, 0, 1, 0, 1]  # 1 for positive, 0 for negative

# Initialize the vectorizer and model
vectorizer = CountVectorizer()
model = LogisticRegression()

# Fit the vectorizer and model
X_train_vectorized = vectorizer.fit_transform(X_train)
model.fit(X_train_vectorized, y_train)

# Initialize stemmer
stemmer = PorterStemmer()

# Custom stopwords function
def custom_stopwords():
    stop_words = set(stopwords.words('english'))
    stop_words.remove("not")  # Ensure 'not' is not removed
    return stop_words

def predict_sentiment(text):
    text = re.sub('[^a-zA-Z ]', '', text)  # Removing non-alphabetic characters
    text = text.lower()  # Convert text to lowercase
    text = text.split()  # Split text into words

    stop_words = custom_stopwords()  # Get custom stopwords that include 'not'
    text = [stemmer.stem(word) for word in text if word not in stop_words]  # Stem and filter stopwords

    text = ' '.join(text)  # Join words back into a single string
    text_vectorized = vectorizer.transform([text])  # Vectorize the text

    # Predict sentiment using the fitted model
    prediction = model.predict(text_vectorized)  # Use the trained model to predict

    if prediction == 0:
        return "Negative"
    else:
        return "Positive"

# Test the function with some sentences
print(predict_sentiment("I hate you"))
print(predict_sentiment("I love you"))


Negative
Positive


In [ ]:
# Save the model
import pickle
pickle.dump(model , open('model.pkl' , 'wb'))

In [ ]:
pickle.dump(vectorizer , open('vectorizer.pkl' , 'wb'))